# <center>**04_data_build_subsets**</center>

### Table of Contents

1. **Notebook Overview**  
    - Builds balanced, reproducible episode-level subsets for Experiments 2–3 (grouped by episode ID).
    - Includes only episodes with both standard projections (RE + RI) and applies workstation-based filtering.
    - Creates comparable splits to evaluate input strategies (single-projection, dual-projection multichannel, episode-level fusion).
    - Restricts to CZC1198HNN and RADIOLOGIA-HP due to sample size; reusable subset-generation pipeline.

2. **Environment Setup and Imports**
   - Import of required libraries  
   - Global paths, constants, and reproducibility settings (random seed)  

3. **Exploratory Data Analysis**
   - Verification of RE/RI availability per episode (**epi_cod**)  
   - Episode-level class balance by station (CZC1198HNN, RADIOLOGIA-HP)  
   - Final counts used to define train/test subset sizes  

4. **Subset Design and Synthesis**  
   - Subset design (target sample sizes per station and split)  
   - Stratified sampling at episode level (balanced by **label_CalTend**)  
   - Subset materialization (copy/save .npy + CSV metadata)  
   - Generation of three input formats:  
     - **1proje_RE** (single-projection RE)  
     - **1proje_RI** (single-projection RI)  
     - **2proje** (RE/RI/padding as 3-channel input)  
   - Logging of missing files and shape mismatches  

5. **Summary and Conclusions**  
   - Final subset inventory and integrity checks  
   - Key constraints (missing images / inconsistent shapes) and impact  
   - Readiness of subsets for downstream training and evaluation (Experiment 2 and 3)  


### **1. Notebook Overview**

This notebook describes the construction of all dataset subsets required for Experiment 2 and 3, which investigates the impact of different projection-based input strategies on the diagnosis of calcifying tendinopathy using convolutional neural networks. Building on the datasets and preprocessing pipeline established in Experiment 2 and 3, this experiment focuses exclusively on clinical episodes that include both standard radiographic projections: external rotation (RE) and internal rotation (RI).

The main methodological objective is to generate strictly comparable, balanced, and reproducible episode-level subsets, ensuring that all evaluated models are trained and tested with equivalent clinical information. Episodes are grouped by clinical episode identifier, filtered by workstation, and balanced by diagnostic class prior to subset generation. This design enables fair comparisons between single-projection models, dual-projection multichannel models, and episode-level fusion strategies, isolating the effect of the input representation from confounding factors related to data composition.

Based on the episode-level class distribution across workstations, Experiment 3 is restricted to the two stations with sufficient sample size to support robust model training and reliable statistical analysis: CZC1198HNN and RADIOLOGIA-HP. The remaining stations contain substantially fewer episodes, which would lead to unstable training and high variance in performance estimates. Consequently, only CZC1198HNN and RADIOLOGIA-HP are used to construct balanced training and test subsets for Experiment 2 and 3.

The subset construction strategy implemented in this notebook reflects a general and reusable pipeline. The same principles of episode-level grouping, class balancing, and station-aware filtering have been applied to the training subsets of other experiments (e.g., Experiments 1 and 4), with minor adaptations to accommodate experiment-specific requirements. This notebook therefore serves as a representative template for balanced subset generation across the entire experimental framework, ensuring methodological consistency while avoiding unnecessary duplication of nearly identical subset-creation notebooks.

### **2. Environment Setup**

In [1]:
# Standard library
import os
import shutil
import warnings

# Numerical and data handling
import numpy as np
import pandas as pd

# Preprocessing utilities
from sklearn.preprocessing import OrdinalEncoder

# Global configuration
warnings.filterwarnings("ignore")


### **3. Exploratory Data Analysis**

In [5]:
# Load labels and metadata
df_data = pd.read_csv("Databases/df_all_data_unique.csv")
print(df_data.shape)

# Select clinical episodes with both standard projections (RE and RI)
df_proj_by_episode = (
    df_data
    .groupby("epi_cod")["proje"]
    .agg(", ".join)
    .reset_index()
)

df_proj_by_episode = df_proj_by_episode[df_proj_by_episode["proje"] == "RE, RI"]

epi_cod_two_projections = df_proj_by_episode["epi_cod"].tolist()
print(f"Episodes with both RE and RI projections: {len(epi_cod_two_projections)}")

# Filter dataset to episodes with two projections
df_two_projections = df_data[df_data["epi_cod"].isin(epi_cod_two_projections)].copy()
print(df_two_projections.shape)

# Episode-level class distribution per workstation
df_episode_labels = (
    df_two_projections
    .groupby(["epi_cod", "station_name"], as_index=False)["label_CalTend"]
    .max()
)

df_counts = (
    df_episode_labels
    .groupby(["station_name", "label_CalTend"])
    .size()
    .reset_index(name="n_episodes")
)

df_pivot = df_counts.pivot_table(
    index="station_name",
    columns="label_CalTend",
    values="n_episodes",
    fill_value=0
)

df_pivot


(4978, 19)
Episodes with both RE and RI projections: 2474
(4948, 19)


label_CalTend,No,Sí
station_name,,
CZC1198HNF,0.0,2.0
CZC1198HNN,489.0,554.0
HRJ SALA 2,6.0,18.0
HRJCDXVILLA,33.0,11.0
HRJCDXVILLA24,26.0,8.0
MININT-VSAMOTD,197.0,170.0
PRIMO R,78.0,17.0
RADIOLOGIA-HP,405.0,460.0


### **4. Subset design and synthesis**

In [58]:
# Build subsets dict 

subsets_dict = {
            'TC_Canon_CZC_2proj_HURJC_train':    ['CZC1198HNN',    878],
            'TC_Canon_RAD_2proj_HURJC_train':   ['RADIOLOGIA-HP', 710],         
            'TC_Canon_CZC_2proj_HURJC_test':     ['CZC1198HNN',     100],
            'TC_Canon_RAD_2proj_HURJC_test':    ['RADIOLOGIA-HP',  100], 
}

In [60]:
# Function to stratify and create subsets


def stratify_subset_exp3(df_data, epi_cod, label, station_name, n_subset, col_rx_id, image_directory, subsets_dir, subset_name, seed=42):
    # Copy dataframe
    df = df_data.copy()

    # Encode binary labels if needed
    if df[label].dtype != 'int':
        encoder = OrdinalEncoder()
        df[label] = encoder.fit_transform(df[[label]]).astype(int)

    # Filter by station name
    df = df[df['station_name'] == station_name]

    # Groupby epi_cod
    df_groupby_epi_cod = df.groupby(epi_cod).first()

    # Select index of given station_name and each class
    idx_station_l0 = df_groupby_epi_cod[df_groupby_epi_cod[label] == 0].index
    idx_station_l1 = df_groupby_epi_cod[df_groupby_epi_cod[label] == 1].index

    # Generate random index
    high_l0 = len(idx_station_l0)
    high_l1 = len(idx_station_l1)

    np.random.seed(seed)
    idx_list_l0 = np.random.choice(high_l0, size=int(n_subset / 2), replace=False)
    idx_list_l1 = np.random.choice(high_l1, size=int(n_subset / 2), replace=False)

    idx_l0 = np.array(idx_station_l0)[idx_list_l0]
    idx_l1 = np.array(idx_station_l1)[idx_list_l1]
    idx_selected = np.concatenate([idx_l0, idx_l1])
    np.random.shuffle(idx_selected)

    df_subset = df_groupby_epi_cod.loc[idx_selected].copy()
    df_subset_rx_level = df[df[epi_cod].isin(idx_selected)].copy()

    X_names = [X_name[:-2] for X_name in df_subset[col_rx_id].values]

    os.makedirs(subsets_dir, exist_ok=True)
    subset_dir = os.path.join(subsets_dir, subset_name)
    os.makedirs(subset_dir, exist_ok=False)

    subset_dir_all = subset_dir.replace('_2proj', '')
    subset_dir_RE = subset_dir.replace('2proj', '1proje_RE')
    subset_dir_RI = subset_dir.replace('2proj', '1proje_RI')

    os.makedirs(subset_dir_all, exist_ok=False)
    os.makedirs(subset_dir_RE, exist_ok=False)
    os.makedirs(subset_dir_RI, exist_ok=False)

    missing = 0
    copied = 0
    missing_epi_cod = []

    for X_name in X_names:
        X_path_RE = os.path.join(image_directory, X_name + 'RE_cropped.npy')
        X_path_RI = os.path.join(image_directory, X_name + 'RI_cropped.npy')

        X_RE_new_path = os.path.join(subset_dir_RE, X_name + 'RE_cropped.npy')
        X_RI_new_path = os.path.join(subset_dir_RI, X_name + 'RI_cropped.npy')

        X_all_RE_new_path = os.path.join(subset_dir_all, X_name + 'RE_cropped.npy')
        X_all_RI_new_path = os.path.join(subset_dir_all, X_name + 'RI_cropped.npy')

        X_2proje_new_path = os.path.join(subset_dir, X_name + '2proje_cropped.npy')

        if os.path.exists(X_path_RE) and os.path.exists(X_path_RI) and not os.path.exists(X_2proje_new_path):
            X_RE = np.load(X_path_RE)
            X_RI = np.load(X_path_RI)
            X_3 = np.zeros_like(X_RI)

            # Comprobación de formas
            if X_RE.shape == X_RI.shape == X_3.shape:
                try:
                    X_2proje = np.stack([X_RE[:, :, 1], X_RI[:, :, 1], X_3[:, :, 1]], axis=-1)
                    np.save(X_2proje_new_path, X_2proje)

                    shutil.copy(X_path_RE, X_RE_new_path)
                    shutil.copy(X_path_RE, X_all_RE_new_path)
                    shutil.copy(X_path_RI, X_RI_new_path)
                    shutil.copy(X_path_RI, X_all_RI_new_path)

                    copied += 1
                except Exception as e:
                    print(f"Error stacking for {X_name}: {e}")
                    missing += 1
                    missing_epi_cod.append(X_name[:-3])
            else:
                print(f"Shape mismatch for {X_name}: RE={X_RE.shape}, RI={X_RI.shape}, Z={X_3.shape}")
                missing += 1
                missing_epi_cod.append(X_name[:-3])
        else:
            if not os.path.exists(X_path_RE):
                print(f'File not exists: {X_path_RE}')
            elif not os.path.exists(X_path_RI):
                print(f'File not exists: {X_path_RI}')
            elif os.path.exists(X_2proje_new_path):
                print(f'File already exists: {X_2proje_new_path}')
            missing += 1
            missing_epi_cod.append(X_name[:-3])

    print(f'Copied:  {copied}  files')
    print(f'Missing: {missing} files')

    # Drop missing images
    df_subset_rx_level = df_subset_rx_level[~df_subset_rx_level[epi_cod].isin(missing_epi_cod)]

    # Save df_subset dataframe
    df_subset_name = os.path.join(subsets_dir, subset_name + '.csv')
    df_subset_rx_level.to_csv(df_subset_name, index=True)

    # Return the remaining dataframe
    df_data_groupby_epi_cod = df_data.groupby(epi_cod).first()
    idx_remaining = np.setdiff1d(np.array(df_data_groupby_epi_cod.index.values), idx_selected)
    df_remaining = df_data[df_data[epi_cod].isin(idx_remaining)].copy()
    df_remaining = df_remaining.reset_index(drop=True)

    print(f'Subset: {subset_name}')
    print(f'Saved:  {subset_dir}')
    print(f'Images: {len(X_names)/2}')
    print('Done!\n')

    return df_remaining
    

In [61]:
# Create subsets 
df_ = df_data.copy()

counter = 0
for subset, params in subsets_dict.items():

    counter += 1
    print(f'Generating subset.... {counter}/{len(subsets_dict)}')
    
    df_ = stratify_subset_exp3 (df_data=df_,
                                epi_cod = 'epi_cod',
                                label='label_CalTend',
                                station_name=params[0],
                                n_subset=params[1],
                                col_rx_id='rx_cod',
                                image_directory='Databases/Images_Xrays_Exp2&3',
                                subsets_dir='Databases/Images_Xrays_Exp2&3',
                                subset_name=subset,
                                seed=42)
    

Generating subset.... 1/4
File not exists: big_volume/Cropped_images_unet/Episodio02343_D_RI_cropped.npy
File not exists: big_volume/Cropped_images_unet/Episodio02127_I_RE_cropped.npy
File not exists: big_volume/Cropped_images_unet/Episodio02437_D_RI_cropped.npy
Shape mismatch for Episodio00479_I_: RE=(300, 300, 3), RI=(201, 300, 3), Z=(201, 300, 3)
File not exists: big_volume/Cropped_images_unet/Episodio03761_D_RI_cropped.npy
File not exists: big_volume/Cropped_images_unet/Episodio02195_D_RE_cropped.npy
File not exists: big_volume/Cropped_images_unet/Episodio00545_I_RI_cropped.npy
File not exists: big_volume/Cropped_images_unet/Episodio02153_D_RE_cropped.npy
Copied:  870  files
Missing: 8 files
Subset: TC_Canon_CZC_2proj_HURJC_train
Saved:  big_volume/prueba/TC_Canon_CZC_2proj_HURJC_train
Images: 439.0
Done!

Generating subset.... 2/4
File not exists: big_volume/Cropped_images_unet/Episodio02150_I_RE_cropped.npy
File not exists: big_volume/Cropped_images_unet/Episodio02181_D_RE_croppe

### **5. Summary and Conclusions**

In this notebook, all datasets required for Experiment 2 and 3 were constructed following a rigorous, episode-centered design. Only clinical episodes containing both standard projections (RE and RI) were included, ensuring that all projection-based strategies were evaluated under identical clinical conditions. This restriction is essential to avoid confounding effects related to incomplete imaging information.

The implemented subset synthesis strategy enabled the generation of three complementary input configurations: single-projection models, dual-projection multichannel models, and global episode-level representations. Balanced sampling by diagnostic class and strict separation by workstation guarantee that subsequent performance differences can be attributed to model architecture and fusion strategy rather than dataset bias or leakage.

Overall, this dataset construction pipeline provides a robust and transparent foundation for Experiment 2 and 3. By aligning episode-level labels, enforcing projection completeness, and carefully managing missing data, the notebook ensures methodological validity and reproducibility. These design choices are critical for drawing reliable conclusions about the added value of multi-projection reasoning in radiographic AI systems.